# Challenge Setup

This notebook creates the schema and tables used in **Challenge: SQL Skills Challenge**.

In [0]:
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = f"challenge_3_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

In [0]:
import random
from pyspark.sql import Row
from datetime import date, timedelta

random.seed(99)

# store_transactions - 60 days of online store sales with some duplicates
regions = ['APAC', 'EMEA', 'Americas']
categories = ['Laptops', 'Phones', 'Accessories', 'Tablets']
channels = ['online', 'in_store']

rows = []
for i in range(400):
    txn_date = date(2024, 2, 1) + timedelta(days=random.randint(0, 59))
    region = random.choice(regions)
    category = random.choice(categories)
    channel = random.choice(channels)
    amount = round(random.uniform(25, 800), 2)
    quantity = random.randint(1, 5)
    
    rows.append(Row(
        transaction_id=5000 + i,
        txn_date=txn_date,
        region=region,
        product_category=category,
        channel=channel,
        amount=amount,
        quantity=quantity
    ))

# Add ~30 intentional duplicates (same txn_id, same date)
for i in range(30):
    orig = rows[random.randint(0, 399)]
    rows.append(Row(
        transaction_id=orig.transaction_id,
        txn_date=orig.txn_date,
        region=orig.region,
        product_category=orig.product_category,
        channel=orig.channel,
        amount=orig.amount,
        quantity=orig.quantity
    ))

df_txn = spark.createDataFrame(rows)
df_txn.write.mode("overwrite").saveAsTable("store_transactions")

count = spark.sql("SELECT COUNT(*) FROM store_transactions").first()[0]

In [0]:
from pyspark.sql import Row

# product_catalog - reference table for JOINs
products = [
    Row(product_category='Laptops', department='Computing', margin_pct=18.0),
    Row(product_category='Phones', department='Mobile', margin_pct=22.0),
    Row(product_category='Accessories', department='Mobile', margin_pct=45.0),
    Row(product_category='Tablets', department='Computing', margin_pct=20.0),
]

df_prod = spark.createDataFrame(products)
df_prod.write.mode("overwrite").saveAsTable("product_catalog")

In [0]:
# Clean up any prior views from earlier challenge attempts
for v in ['v_clean_transactions', 'v_daily_summary']:
    spark.sql(f"DROP VIEW IF EXISTS {v}")

print("CHALLENGE SETUP COMPLETE")
print(f"")
print(f"Catalog:  {catalog_name}")
print(f"Schema:   {schema_name}")
print(f"")
print(f"Tables:")
print(f"store_transactions  - 430 rows (60 days, 3 regions, includes duplicates)")
print(f"product_catalog     - 4 rows (category reference with margin %)")